In [1]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb
# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)

# 1. Base 데이터 (v0) 로드
path_v0 = os.path.join(DATA_DIR, "featured_v0.parquet")
print(f"[LOAD] Base: {path_v0}")
final_data = pd.read_parquet(path_v0)
print(f"   -> Base Shape: {final_data.shape}")

# 2. v1 ~ v4 순회하며 새로운 컬럼만 병합
versions = ["v1", "v2", "v3", "v4"]

for ver in versions:
    path_ver = os.path.join(DATA_DIR, f"featured_{ver}.parquet")
    if os.path.exists(path_ver):
        print(f"[MERGE] Checking {ver}...")
        temp_df = pd.read_parquet(path_ver)
        
        # v0(현재 final_data)에 없는 컬럼만 추출 (즉, 추가된 A, B, C, D 부분)
        new_cols = [c for c in temp_df.columns if c not in final_data.columns]
        
        if new_cols:
            # 인덱스 기준 병합 (axis=1). 행 순서가 동일하다고 가정합니다.
            # 만약 행 순서가 다르다면 merge(on='id')를 써야 하지만, 
            # 보통 FE 파이프라인 결과물은 순서가 같습니다.
            final_data = pd.concat([final_data, temp_df[new_cols]], axis=1)
            print(f"   -> Added {len(new_cols)} features from {ver} (ex: {new_cols[:3]}...)")
        else:
            print(f"   -> No new features in {ver}")
            
        # 메모리 정리
        del temp_df
        gc.collect()
    else:
        print(f"[WARN] File not found: {path_ver}")

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=OPERATING_THR, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0) #+ cols_v1)# + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_3d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).to_numpy()
    Y = df[TARGET].fillna(0).astype(np.int32).to_numpy()
    X_3d = X.reshape(X.shape[0], X.shape[1], 1)  # (N, F, 1)
    return X_3d, Y

Xtr, Ytr = prepare_data_3d(train_df, feat_cols)
Xva, Yva = prepare_data_3d(val_df, feat_cols)
Xte, Yte = prepare_data_3d(test_df, feat_cols)

print("=== Data Shapes ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")


# =========================
# RETRAIN WITH BEST PARAMS (전체 train_df 사용)
# =========================
print("\n[Final] Retraining model with best hyperparameters...")
best_params = {
      "lr": 0.00017674389197537286,
      "batch_size": 32,
      "lstm_units1": 256,
      "lstm_units2": 64,
      "dense_units": 64,
      "dropout_rate": 0.3653765991273793,
      "l2": 0.0007329716219431157
    }

tf.keras.backend.clear_session()
inp = layers.Input(shape=(n_feat, 1))
x = layers.LSTM(best_params["lstm_units1"], return_sequences=True,
                kernel_regularizer=regularizers.l2(best_params["l2"]))(inp)
x = layers.LSTM(best_params["lstm_units2"], return_sequences=False,
                kernel_regularizer=regularizers.l2(best_params["l2"]))(x)
x = layers.Dense(best_params["dense_units"], activation='relu',
                 kernel_regularizer=regularizers.l2(best_params["l2"]))(x)
x = layers.Dropout(best_params["dropout_rate"])(x)
out = layers.Dense(1, activation="sigmoid")(x)

best_model = models.Model(inp, out)
opt = optimizers.Adam(learning_rate=best_params["lr"])

# metrics에 AUC 추가
best_model.compile(optimizer=opt, loss="binary_crossentropy", 
                   metrics=[tf.keras.metrics.AUC(name='auc')])

# 1. 콜백 정의 (fit보다 먼저 나와야 함)
es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
rlr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                  patience=2, min_lr=1e-6)

# 2. 모델 학습 (괄호 수정 및 중복 제거 완료)
history = best_model.fit(
    Xtr, Ytr,
    validation_data=(Xva, Yva),
    epochs=EPOCHS,
    batch_size=best_params["batch_size"],
    verbose=2,
    callbacks=[es, rlr]
)

# =========================
# FINAL EVALUATION (수동 metric)
# =========================
y_prob = best_model.predict(Xte, verbose=0).ravel()
y_true = Yte
n_features_used = Xte.shape[1]

# 평가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, thr=OPERATING_THR, n_features=n_features_used
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    "best_params": best_params
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)

2025-12-17 06:45:11.419340: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-17 06:45:11.434812: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765921511.453667  124738 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765921511.459505  124738 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765921511.474654  124738 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

[LOAD] Base: ./featured_v0.parquet
   -> Base Shape: (433520, 62)
[MERGE] Checking v1...


   -> Added 3 features from v1 (ex: ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']...)
[MERGE] Checking v2...


   -> Added 7 features from v2 (ex: ['feat_hour_shift', 'feat_dow_shift', 'feat_payday_bump']...)
[MERGE] Checking v3...


   -> Added 45 features from v3 (ex: ['is_hard_bounced', 'hard_bounced_at', 'is_soft_bounced']...)
[MERGE] Checking v4...


...

Epoch 49/50


2188/2188 - 27s - 12ms/step - auc: 0.9611 - loss: 0.1298 - val_auc: 0.9580 - val_loss: 0.1352 - learning_rate: 1.0000e-06


Epoch 50/50


2188/2188 - 27s - 12ms/step - auc: 0.9615 - loss: 0.1298 - val_auc: 0.9578 - val_loss: 0.1352 - learning_rate: 1.0000e-06


{
  "metrics": {
    "accuracy": 0.9459999999999369,
    "precision": 0.8536345776023049,
    "recall": 0.5679738562087792,
    "specificity": 0.9889383815886422,
    "f1": 0.6821036101946928,
    "roc_auc": 0.9630219175024025,
    "n_eval": 15000,
    "pos_rate_eval": 0.102,
    "aic": 3870.6242809814667,
    "bic": 4182.872305664925,
    "log_likelihood": -1894.3121404907333,
    "n_features": 41,
    "mode": "STRICT_CAUSAL",
    "split": "RANDOM_ROW",
    "best_params": {
      "lr": 0.00017674389197537286,
      "batch_size": 32,
      "lstm_units1": 256,
      "lstm_units2": 64,
      "dense_units": 64,
      "dropout_rate": 0.3653765991273793,
      "l2": 0.0007329716219431157
    }
  }
}
confusion_matrix [tn fp fn tp]: 13321 149 661 869


In [2]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb
# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)

# 1. Base 데이터 (v0) 로드
path_v0 = os.path.join(DATA_DIR, "featured_v0.parquet")
print(f"[LOAD] Base: {path_v0}")
final_data = pd.read_parquet(path_v0)
print(f"   -> Base Shape: {final_data.shape}")

# 2. v1 ~ v4 순회하며 새로운 컬럼만 병합
versions = ["v1", "v2", "v3", "v4"]

for ver in versions:
    path_ver = os.path.join(DATA_DIR, f"featured_{ver}.parquet")
    if os.path.exists(path_ver):
        print(f"[MERGE] Checking {ver}...")
        temp_df = pd.read_parquet(path_ver)
        
        # v0(현재 final_data)에 없는 컬럼만 추출 (즉, 추가된 A, B, C, D 부분)
        new_cols = [c for c in temp_df.columns if c not in final_data.columns]
        
        if new_cols:
            # 인덱스 기준 병합 (axis=1). 행 순서가 동일하다고 가정합니다.
            # 만약 행 순서가 다르다면 merge(on='id')를 써야 하지만, 
            # 보통 FE 파이프라인 결과물은 순서가 같습니다.
            final_data = pd.concat([final_data, temp_df[new_cols]], axis=1)
            print(f"   -> Added {len(new_cols)} features from {ver} (ex: {new_cols[:3]}...)")
        else:
            print(f"   -> No new features in {ver}")
            
        # 메모리 정리
        del temp_df
        gc.collect()
    else:
        print(f"[WARN] File not found: {path_ver}")

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=OPERATING_THR, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0+ cols_v1)# + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_3d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).to_numpy()
    Y = df[TARGET].fillna(0).astype(np.int32).to_numpy()
    X_3d = X.reshape(X.shape[0], X.shape[1], 1)  # (N, F, 1)
    return X_3d, Y

Xtr, Ytr = prepare_data_3d(train_df, feat_cols)
Xva, Yva = prepare_data_3d(val_df, feat_cols)
Xte, Yte = prepare_data_3d(test_df, feat_cols)

print("=== Data Shapes ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")


# =========================
# RETRAIN WITH BEST PARAMS (전체 train_df 사용)
# =========================
print("\n[Final] Retraining model with best hyperparameters...")
best_params = {
      "lr": 0.00017674389197537286,
      "batch_size": 32,
      "lstm_units1": 256,
      "lstm_units2": 64,
      "dense_units": 64,
      "dropout_rate": 0.3653765991273793,
      "l2": 0.0007329716219431157
    }

tf.keras.backend.clear_session()
inp = layers.Input(shape=(n_feat, 1))
x = layers.LSTM(best_params["lstm_units1"], return_sequences=True,
                kernel_regularizer=regularizers.l2(best_params["l2"]))(inp)
x = layers.LSTM(best_params["lstm_units2"], return_sequences=False,
                kernel_regularizer=regularizers.l2(best_params["l2"]))(x)
x = layers.Dense(best_params["dense_units"], activation='relu',
                 kernel_regularizer=regularizers.l2(best_params["l2"]))(x)
x = layers.Dropout(best_params["dropout_rate"])(x)
out = layers.Dense(1, activation="sigmoid")(x)

best_model = models.Model(inp, out)
opt = optimizers.Adam(learning_rate=best_params["lr"])

# metrics에 AUC 추가
best_model.compile(optimizer=opt, loss="binary_crossentropy", 
                   metrics=[tf.keras.metrics.AUC(name='auc')])

# 1. 콜백 정의 (fit보다 먼저 나와야 함)
es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
rlr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                  patience=2, min_lr=1e-6)

# 2. 모델 학습 (괄호 수정 및 중복 제거 완료)
history = best_model.fit(
    Xtr, Ytr,
    validation_data=(Xva, Yva),
    epochs=EPOCHS,
    batch_size=best_params["batch_size"],
    verbose=2,
    callbacks=[es, rlr]
)

# =========================
# FINAL EVALUATION (수동 metric)
# =========================
y_prob = best_model.predict(Xte, verbose=0).ravel()
y_true = Yte
n_features_used = Xte.shape[1]

# 평가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, thr=OPERATING_THR, n_features=n_features_used
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    "best_params": best_params
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)

[LOAD] Base: ./featured_v0.parquet
   -> Base Shape: (433520, 62)
[MERGE] Checking v1...


   -> Added 3 features from v1 (ex: ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']...)
[MERGE] Checking v2...


   -> Added 7 features from v2 (ex: ['feat_hour_shift', 'feat_dow_shift', 'feat_payday_bump']...)
[MERGE] Checking v3...


   -> Added 45 features from v3 (ex: ['is_hard_bounced', 'hard_bounced_at', 'is_soft_bounced']...)
[MERGE] Checking v4...


   -> Added 7 features from v4 (ex: ['tn_sent_local', 'feat_topic_novelty', 'topic_N7']...)

[FINAL DATA] Total Shape: (579551, 124)


...

Epoch 42/50


2188/2188 - 26s - 12ms/step - auc: 0.9663 - loss: 0.1220 - val_auc: 0.9635 - val_loss: 0.1291 - learning_rate: 2.7616e-06


Epoch 43/50


2188/2188 - 26s - 12ms/step - auc: 0.9669 - loss: 0.1216 - val_auc: 0.9629 - val_loss: 0.1293 - learning_rate: 1.3808e-06


{
  "metrics": {
    "accuracy": 0.9482666666666034,
    "precision": 0.8465073529403985,
    "recall": 0.6019607843133321,
    "specificity": 0.9876020786933194,
    "f1": 0.7035905266336533,
    "roc_auc": 0.9655473067722039,
    "n_eval": 15000,
    "pos_rate_eval": 0.102,
    "aic": 3739.4341403208837,
    "bic": 4074.529581444595,
    "log_likelihood": -1825.7170701604418,
    "n_features": 44,
    "mode": "STRICT_CAUSAL",
    "split": "RANDOM_ROW",
    "best_params": {
      "lr": 0.00017674389197537286,
      "batch_size": 32,
      "lstm_units1": 256,
      "lstm_units2": 64,
      "dense_units": 64,
      "dropout_rate": 0.3653765991273793,
      "l2": 0.0007329716219431157
    }
  }
}
confusion_matrix [tn fp fn tp]: 13303 167 609 921


In [3]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb
# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)

# 1. Base 데이터 (v0) 로드
path_v0 = os.path.join(DATA_DIR, "featured_v0.parquet")
print(f"[LOAD] Base: {path_v0}")
final_data = pd.read_parquet(path_v0)
print(f"   -> Base Shape: {final_data.shape}")

# 2. v1 ~ v4 순회하며 새로운 컬럼만 병합
versions = ["v1", "v2", "v3", "v4"]

for ver in versions:
    path_ver = os.path.join(DATA_DIR, f"featured_{ver}.parquet")
    if os.path.exists(path_ver):
        print(f"[MERGE] Checking {ver}...")
        temp_df = pd.read_parquet(path_ver)
        
        # v0(현재 final_data)에 없는 컬럼만 추출 (즉, 추가된 A, B, C, D 부분)
        new_cols = [c for c in temp_df.columns if c not in final_data.columns]
        
        if new_cols:
            # 인덱스 기준 병합 (axis=1). 행 순서가 동일하다고 가정합니다.
            # 만약 행 순서가 다르다면 merge(on='id')를 써야 하지만, 
            # 보통 FE 파이프라인 결과물은 순서가 같습니다.
            final_data = pd.concat([final_data, temp_df[new_cols]], axis=1)
            print(f"   -> Added {len(new_cols)} features from {ver} (ex: {new_cols[:3]}...)")
        else:
            print(f"   -> No new features in {ver}")
            
        # 메모리 정리
        del temp_df
        gc.collect()
    else:
        print(f"[WARN] File not found: {path_ver}")

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=OPERATING_THR, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0+ cols_v2)# + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_3d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).to_numpy()
    Y = df[TARGET].fillna(0).astype(np.int32).to_numpy()
    X_3d = X.reshape(X.shape[0], X.shape[1], 1)  # (N, F, 1)
    return X_3d, Y

Xtr, Ytr = prepare_data_3d(train_df, feat_cols)
Xva, Yva = prepare_data_3d(val_df, feat_cols)
Xte, Yte = prepare_data_3d(test_df, feat_cols)

print("=== Data Shapes ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")


# =========================
# RETRAIN WITH BEST PARAMS (전체 train_df 사용)
# =========================
print("\n[Final] Retraining model with best hyperparameters...")
best_params = {
      "lr": 0.00017674389197537286,
      "batch_size": 32,
      "lstm_units1": 256,
      "lstm_units2": 64,
      "dense_units": 64,
      "dropout_rate": 0.3653765991273793,
      "l2": 0.0007329716219431157
    }

tf.keras.backend.clear_session()
inp = layers.Input(shape=(n_feat, 1))
x = layers.LSTM(best_params["lstm_units1"], return_sequences=True,
                kernel_regularizer=regularizers.l2(best_params["l2"]))(inp)
x = layers.LSTM(best_params["lstm_units2"], return_sequences=False,
                kernel_regularizer=regularizers.l2(best_params["l2"]))(x)
x = layers.Dense(best_params["dense_units"], activation='relu',
                 kernel_regularizer=regularizers.l2(best_params["l2"]))(x)
x = layers.Dropout(best_params["dropout_rate"])(x)
out = layers.Dense(1, activation="sigmoid")(x)

best_model = models.Model(inp, out)
opt = optimizers.Adam(learning_rate=best_params["lr"])

# metrics에 AUC 추가
best_model.compile(optimizer=opt, loss="binary_crossentropy", 
                   metrics=[tf.keras.metrics.AUC(name='auc')])

# 1. 콜백 정의 (fit보다 먼저 나와야 함)
es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
rlr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                  patience=2, min_lr=1e-6)

# 2. 모델 학습 (괄호 수정 및 중복 제거 완료)
history = best_model.fit(
    Xtr, Ytr,
    validation_data=(Xva, Yva),
    epochs=EPOCHS,
    batch_size=best_params["batch_size"],
    verbose=2,
    callbacks=[es, rlr]
)

# =========================
# FINAL EVALUATION (수동 metric)
# =========================
y_prob = best_model.predict(Xte, verbose=0).ravel()
y_true = Yte
n_features_used = Xte.shape[1]

# 평가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, thr=OPERATING_THR, n_features=n_features_used
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    "best_params": best_params
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)

[LOAD] Base: ./featured_v0.parquet
   -> Base Shape: (433520, 62)
[MERGE] Checking v1...


   -> Added 3 features from v1 (ex: ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']...)
[MERGE] Checking v2...


   -> Added 7 features from v2 (ex: ['feat_hour_shift', 'feat_dow_shift', 'feat_payday_bump']...)
[MERGE] Checking v3...


   -> Added 45 features from v3 (ex: ['is_hard_bounced', 'hard_bounced_at', 'is_soft_bounced']...)
[MERGE] Checking v4...


   -> Added 7 features from v4 (ex: ['tn_sent_local', 'feat_topic_novelty', 'topic_N7']...)

[FINAL DATA] Total Shape: (579551, 124)


...

Epoch 49/50


2188/2188 - 26s - 12ms/step - auc: 0.9754 - loss: 0.1069 - val_auc: 0.9706 - val_loss: 0.1172 - learning_rate: 1.1046e-05


Epoch 50/50


2188/2188 - 26s - 12ms/step - auc: 0.9758 - loss: 0.1062 - val_auc: 0.9715 - val_loss: 0.1155 - learning_rate: 5.5232e-06


{
  "metrics": {
    "accuracy": 0.9547999999999364,
    "precision": 0.8526490066218108,
    "recall": 0.6732026143786449,
    "specificity": 0.9867854491461776,
    "f1": 0.7523739951236045,
    "roc_auc": 0.9741478279011732,
    "n_eval": 15000,
    "pos_rate_eval": 0.102,
    "aic": 3348.71633745682,
    "bic": 3706.6591950207844,
    "log_likelihood": -1627.35816872841,
    "n_features": 47,
    "mode": "STRICT_CAUSAL",
    "split": "RANDOM_ROW",
    "best_params": {
      "lr": 0.00017674389197537286,
      "batch_size": 32,
      "lstm_units1": 256,
      "lstm_units2": 64,
      "dense_units": 64,
      "dropout_rate": 0.3653765991273793,
      "l2": 0.0007329716219431157
    }
  }
}
confusion_matrix [tn fp fn tp]: 13292 178 500 1030


In [4]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb
# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)

# 1. Base 데이터 (v0) 로드
path_v0 = os.path.join(DATA_DIR, "featured_v0.parquet")
print(f"[LOAD] Base: {path_v0}")
final_data = pd.read_parquet(path_v0)
print(f"   -> Base Shape: {final_data.shape}")

# 2. v1 ~ v4 순회하며 새로운 컬럼만 병합
versions = ["v1", "v2", "v3", "v4"]

for ver in versions:
    path_ver = os.path.join(DATA_DIR, f"featured_{ver}.parquet")
    if os.path.exists(path_ver):
        print(f"[MERGE] Checking {ver}...")
        temp_df = pd.read_parquet(path_ver)
        
        # v0(현재 final_data)에 없는 컬럼만 추출 (즉, 추가된 A, B, C, D 부분)
        new_cols = [c for c in temp_df.columns if c not in final_data.columns]
        
        if new_cols:
            # 인덱스 기준 병합 (axis=1). 행 순서가 동일하다고 가정합니다.
            # 만약 행 순서가 다르다면 merge(on='id')를 써야 하지만, 
            # 보통 FE 파이프라인 결과물은 순서가 같습니다.
            final_data = pd.concat([final_data, temp_df[new_cols]], axis=1)
            print(f"   -> Added {len(new_cols)} features from {ver} (ex: {new_cols[:3]}...)")
        else:
            print(f"   -> No new features in {ver}")
            
        # 메모리 정리
        del temp_df
        gc.collect()
    else:
        print(f"[WARN] File not found: {path_ver}")

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=OPERATING_THR, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0+ cols_v3)# + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_3d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).to_numpy()
    Y = df[TARGET].fillna(0).astype(np.int32).to_numpy()
    X_3d = X.reshape(X.shape[0], X.shape[1], 1)  # (N, F, 1)
    return X_3d, Y

Xtr, Ytr = prepare_data_3d(train_df, feat_cols)
Xva, Yva = prepare_data_3d(val_df, feat_cols)
Xte, Yte = prepare_data_3d(test_df, feat_cols)

print("=== Data Shapes ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")


# =========================
# RETRAIN WITH BEST PARAMS (전체 train_df 사용)
# =========================
print("\n[Final] Retraining model with best hyperparameters...")
best_params = {
      "lr": 0.00017674389197537286,
      "batch_size": 32,
      "lstm_units1": 256,
      "lstm_units2": 64,
      "dense_units": 64,
      "dropout_rate": 0.3653765991273793,
      "l2": 0.0007329716219431157
    }

tf.keras.backend.clear_session()
inp = layers.Input(shape=(n_feat, 1))
x = layers.LSTM(best_params["lstm_units1"], return_sequences=True,
                kernel_regularizer=regularizers.l2(best_params["l2"]))(inp)
x = layers.LSTM(best_params["lstm_units2"], return_sequences=False,
                kernel_regularizer=regularizers.l2(best_params["l2"]))(x)
x = layers.Dense(best_params["dense_units"], activation='relu',
                 kernel_regularizer=regularizers.l2(best_params["l2"]))(x)
x = layers.Dropout(best_params["dropout_rate"])(x)
out = layers.Dense(1, activation="sigmoid")(x)

best_model = models.Model(inp, out)
opt = optimizers.Adam(learning_rate=best_params["lr"])

# metrics에 AUC 추가
best_model.compile(optimizer=opt, loss="binary_crossentropy", 
                   metrics=[tf.keras.metrics.AUC(name='auc')])

# 1. 콜백 정의 (fit보다 먼저 나와야 함)
es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
rlr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                  patience=2, min_lr=1e-6)

# 2. 모델 학습 (괄호 수정 및 중복 제거 완료)
history = best_model.fit(
    Xtr, Ytr,
    validation_data=(Xva, Yva),
    epochs=EPOCHS,
    batch_size=best_params["batch_size"],
    verbose=2,
    callbacks=[es, rlr]
)

# =========================
# FINAL EVALUATION (수동 metric)
# =========================
y_prob = best_model.predict(Xte, verbose=0).ravel()
y_true = Yte
n_features_used = Xte.shape[1]

# 평가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, thr=OPERATING_THR, n_features=n_features_used
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    "best_params": best_params
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)

[LOAD] Base: ./featured_v0.parquet
   -> Base Shape: (433520, 62)
[MERGE] Checking v1...


   -> Added 3 features from v1 (ex: ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']...)


[MERGE] Checking v2...


   -> Added 7 features from v2 (ex: ['feat_hour_shift', 'feat_dow_shift', 'feat_payday_bump']...)
[MERGE] Checking v3...


   -> Added 45 features from v3 (ex: ['is_hard_bounced', 'hard_bounced_at', 'is_soft_bounced']...)
[MERGE] Checking v4...


...

Epoch 34/50


2188/2188 - 27s - 13ms/step - auc: 0.9602 - loss: 0.1313 - val_auc: 0.9545 - val_loss: 0.1405 - learning_rate: 2.7616e-06


Epoch 35/50


2188/2188 - 27s - 12ms/step - auc: 0.9602 - loss: 0.1312 - val_auc: 0.9545 - val_loss: 0.1407 - learning_rate: 1.3808e-06


{
  "metrics": {
    "accuracy": 0.9456666666666036,
    "precision": 0.8408007626302757,
    "recall": 0.5764705882349174,
    "specificity": 0.9876020786933194,
    "f1": 0.6839860406180639,
    "roc_auc": 0.9599387163923285,
    "n_eval": 15000,
    "pos_rate_eval": 0.102,
    "aic": 4033.487080176442,
    "bic": 4414.277354180659,
    "log_likelihood": -1966.743540088221,
    "n_features": 50,
    "mode": "STRICT_CAUSAL",
    "split": "RANDOM_ROW",
    "best_params": {
      "lr": 0.00017674389197537286,
      "batch_size": 32,
      "lstm_units1": 256,
      "lstm_units2": 64,
      "dense_units": 64,
      "dropout_rate": 0.3653765991273793,
      "l2": 0.0007329716219431157
    }
  }
}
confusion_matrix [tn fp fn tp]: 13303 167 648 882


In [5]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb
# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)

# 1. Base 데이터 (v0) 로드
path_v0 = os.path.join(DATA_DIR, "featured_v0.parquet")
print(f"[LOAD] Base: {path_v0}")
final_data = pd.read_parquet(path_v0)
print(f"   -> Base Shape: {final_data.shape}")

# 2. v1 ~ v4 순회하며 새로운 컬럼만 병합
versions = ["v1", "v2", "v3", "v4"]

for ver in versions:
    path_ver = os.path.join(DATA_DIR, f"featured_{ver}.parquet")
    if os.path.exists(path_ver):
        print(f"[MERGE] Checking {ver}...")
        temp_df = pd.read_parquet(path_ver)
        
        # v0(현재 final_data)에 없는 컬럼만 추출 (즉, 추가된 A, B, C, D 부분)
        new_cols = [c for c in temp_df.columns if c not in final_data.columns]
        
        if new_cols:
            # 인덱스 기준 병합 (axis=1). 행 순서가 동일하다고 가정합니다.
            # 만약 행 순서가 다르다면 merge(on='id')를 써야 하지만, 
            # 보통 FE 파이프라인 결과물은 순서가 같습니다.
            final_data = pd.concat([final_data, temp_df[new_cols]], axis=1)
            print(f"   -> Added {len(new_cols)} features from {ver} (ex: {new_cols[:3]}...)")
        else:
            print(f"   -> No new features in {ver}")
            
        # 메모리 정리
        del temp_df
        gc.collect()
    else:
        print(f"[WARN] File not found: {path_ver}")

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=OPERATING_THR, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0+ cols_v4)# + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_3d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).to_numpy()
    Y = df[TARGET].fillna(0).astype(np.int32).to_numpy()
    X_3d = X.reshape(X.shape[0], X.shape[1], 1)  # (N, F, 1)
    return X_3d, Y

Xtr, Ytr = prepare_data_3d(train_df, feat_cols)
Xva, Yva = prepare_data_3d(val_df, feat_cols)
Xte, Yte = prepare_data_3d(test_df, feat_cols)

print("=== Data Shapes ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")


# =========================
# RETRAIN WITH BEST PARAMS (전체 train_df 사용)
# =========================
print("\n[Final] Retraining model with best hyperparameters...")
best_params = {
      "lr": 0.00017674389197537286,
      "batch_size": 32,
      "lstm_units1": 256,
      "lstm_units2": 64,
      "dense_units": 64,
      "dropout_rate": 0.3653765991273793,
      "l2": 0.0007329716219431157
    }

tf.keras.backend.clear_session()
inp = layers.Input(shape=(n_feat, 1))
x = layers.LSTM(best_params["lstm_units1"], return_sequences=True,
                kernel_regularizer=regularizers.l2(best_params["l2"]))(inp)
x = layers.LSTM(best_params["lstm_units2"], return_sequences=False,
                kernel_regularizer=regularizers.l2(best_params["l2"]))(x)
x = layers.Dense(best_params["dense_units"], activation='relu',
                 kernel_regularizer=regularizers.l2(best_params["l2"]))(x)
x = layers.Dropout(best_params["dropout_rate"])(x)
out = layers.Dense(1, activation="sigmoid")(x)

best_model = models.Model(inp, out)
opt = optimizers.Adam(learning_rate=best_params["lr"])

# metrics에 AUC 추가
best_model.compile(optimizer=opt, loss="binary_crossentropy", 
                   metrics=[tf.keras.metrics.AUC(name='auc')])

# 1. 콜백 정의 (fit보다 먼저 나와야 함)
es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
rlr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                  patience=2, min_lr=1e-6)

# 2. 모델 학습 (괄호 수정 및 중복 제거 완료)
history = best_model.fit(
    Xtr, Ytr,
    validation_data=(Xva, Yva),
    epochs=EPOCHS,
    batch_size=best_params["batch_size"],
    verbose=2,
    callbacks=[es, rlr]
)

# =========================
# FINAL EVALUATION (수동 metric)
# =========================
y_prob = best_model.predict(Xte, verbose=0).ravel()
y_true = Yte
n_features_used = Xte.shape[1]

# 평가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, thr=OPERATING_THR, n_features=n_features_used
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    "best_params": best_params
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)

[LOAD] Base: ./featured_v0.parquet
   -> Base Shape: (433520, 62)
[MERGE] Checking v1...


   -> Added 3 features from v1 (ex: ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']...)


[MERGE] Checking v2...
   -> Added 7 features from v2 (ex: ['feat_hour_shift', 'feat_dow_shift', 'feat_payday_bump']...)


[MERGE] Checking v3...


   -> Added 45 features from v3 (ex: ['is_hard_bounced', 'hard_bounced_at', 'is_soft_bounced']...)
[MERGE] Checking v4...


...

Epoch 42/50


2188/2188 - 26s - 12ms/step - auc: 0.9603 - loss: 0.1330 - val_auc: 0.9554 - val_loss: 0.1412 - learning_rate: 2.7616e-06


Epoch 43/50


2188/2188 - 26s - 12ms/step - auc: 0.9602 - loss: 0.1331 - val_auc: 0.9554 - val_loss: 0.1411 - learning_rate: 1.3808e-06


{
  "metrics": {
    "accuracy": 0.9449333333332703,
    "precision": 0.8352380952372997,
    "recall": 0.5732026143787103,
    "specificity": 0.9871566443948784,
    "f1": 0.6798449607570897,
    "roc_auc": 0.9606104584867307,
    "n_eval": 15000,
    "pos_rate_eval": 0.102,
    "aic": 3998.6298380129842,
    "bic": 4348.956890096864,
    "log_likelihood": -1953.3149190064921,
    "n_features": 46,
    "mode": "STRICT_CAUSAL",
    "split": "RANDOM_ROW",
    "best_params": {
      "lr": 0.00017674389197537286,
      "batch_size": 32,
      "lstm_units1": 256,
      "lstm_units2": 64,
      "dense_units": 64,
      "dropout_rate": 0.3653765991273793,
      "l2": 0.0007329716219431157
    }
  }
}
confusion_matrix [tn fp fn tp]: 13297 173 653 877


In [6]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb
# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)

# 1. Base 데이터 (v0) 로드
path_v0 = os.path.join(DATA_DIR, "featured_v0.parquet")
print(f"[LOAD] Base: {path_v0}")
final_data = pd.read_parquet(path_v0)
print(f"   -> Base Shape: {final_data.shape}")

# 2. v1 ~ v4 순회하며 새로운 컬럼만 병합
versions = ["v1", "v2", "v3", "v4"]

for ver in versions:
    path_ver = os.path.join(DATA_DIR, f"featured_{ver}.parquet")
    if os.path.exists(path_ver):
        print(f"[MERGE] Checking {ver}...")
        temp_df = pd.read_parquet(path_ver)
        
        # v0(현재 final_data)에 없는 컬럼만 추출 (즉, 추가된 A, B, C, D 부분)
        new_cols = [c for c in temp_df.columns if c not in final_data.columns]
        
        if new_cols:
            # 인덱스 기준 병합 (axis=1). 행 순서가 동일하다고 가정합니다.
            # 만약 행 순서가 다르다면 merge(on='id')를 써야 하지만, 
            # 보통 FE 파이프라인 결과물은 순서가 같습니다.
            final_data = pd.concat([final_data, temp_df[new_cols]], axis=1)
            print(f"   -> Added {len(new_cols)} features from {ver} (ex: {new_cols[:3]}...)")
        else:
            print(f"   -> No new features in {ver}")
            
        # 메모리 정리
        del temp_df
        gc.collect()
    else:
        print(f"[WARN] File not found: {path_ver}")

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=OPERATING_THR, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0+ cols_v1 + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_3d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).to_numpy()
    Y = df[TARGET].fillna(0).astype(np.int32).to_numpy()
    X_3d = X.reshape(X.shape[0], X.shape[1], 1)  # (N, F, 1)
    return X_3d, Y

Xtr, Ytr = prepare_data_3d(train_df, feat_cols)
Xva, Yva = prepare_data_3d(val_df, feat_cols)
Xte, Yte = prepare_data_3d(test_df, feat_cols)

print("=== Data Shapes ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")


# =========================
# RETRAIN WITH BEST PARAMS (전체 train_df 사용)
# =========================
print("\n[Final] Retraining model with best hyperparameters...")
best_params = {
      "lr": 0.00017674389197537286,
      "batch_size": 32,
      "lstm_units1": 256,
      "lstm_units2": 64,
      "dense_units": 64,
      "dropout_rate": 0.3653765991273793,
      "l2": 0.0007329716219431157
    }

tf.keras.backend.clear_session()
inp = layers.Input(shape=(n_feat, 1))
x = layers.LSTM(best_params["lstm_units1"], return_sequences=True,
                kernel_regularizer=regularizers.l2(best_params["l2"]))(inp)
x = layers.LSTM(best_params["lstm_units2"], return_sequences=False,
                kernel_regularizer=regularizers.l2(best_params["l2"]))(x)
x = layers.Dense(best_params["dense_units"], activation='relu',
                 kernel_regularizer=regularizers.l2(best_params["l2"]))(x)
x = layers.Dropout(best_params["dropout_rate"])(x)
out = layers.Dense(1, activation="sigmoid")(x)

best_model = models.Model(inp, out)
opt = optimizers.Adam(learning_rate=best_params["lr"])

# metrics에 AUC 추가
best_model.compile(optimizer=opt, loss="binary_crossentropy", 
                   metrics=[tf.keras.metrics.AUC(name='auc')])

# 1. 콜백 정의 (fit보다 먼저 나와야 함)
es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
rlr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                  patience=2, min_lr=1e-6)

# 2. 모델 학습 (괄호 수정 및 중복 제거 완료)
history = best_model.fit(
    Xtr, Ytr,
    validation_data=(Xva, Yva),
    epochs=EPOCHS,
    batch_size=best_params["batch_size"],
    verbose=2,
    callbacks=[es, rlr]
)

# =========================
# FINAL EVALUATION (수동 metric)
# =========================
y_prob = best_model.predict(Xte, verbose=0).ravel()
y_true = Yte
n_features_used = Xte.shape[1]

# 평가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, thr=OPERATING_THR, n_features=n_features_used
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    "best_params": best_params
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)

[LOAD] Base: ./featured_v0.parquet
   -> Base Shape: (433520, 62)
[MERGE] Checking v1...


   -> Added 3 features from v1 (ex: ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']...)


[MERGE] Checking v2...


   -> Added 7 features from v2 (ex: ['feat_hour_shift', 'feat_dow_shift', 'feat_payday_bump']...)


[MERGE] Checking v3...


...

Epoch 10/50


2188/2188 - 27s - 12ms/step - auc: 0.7465 - loss: 0.2376 - val_auc: 0.7640 - val_loss: 0.2447 - learning_rate: 8.8372e-05


Epoch 11/50


2188/2188 - 27s - 12ms/step - auc: 0.7504 - loss: 0.2356 - val_auc: 0.7693 - val_loss: 0.2401 - learning_rate: 4.4186e-05


{
  "metrics": {
    "accuracy": 0.9318666666666046,
    "precision": 0.8802395209567662,
    "recall": 0.3843137254899449,
    "specificity": 0.9940608760207131,
    "f1": 0.5350318467101717,
    "roc_auc": 0.765149812461429,
    "n_eval": 15000,
    "pos_rate_eval": 0.102,
    "aic": 7051.50711882752,
    "bic": 7538.918669552918,
    "log_likelihood": -3461.75355941376,
    "n_features": 64,
    "mode": "STRICT_CAUSAL",
    "split": "RANDOM_ROW",
    "best_params": {
      "lr": 0.00017674389197537286,
      "batch_size": 32,
      "lstm_units1": 256,
      "lstm_units2": 64,
      "dense_units": 64,
      "dropout_rate": 0.3653765991273793,
      "l2": 0.0007329716219431157
    }
  }
}
confusion_matrix [tn fp fn tp]: 13390 80 942 588
